# Model Deployment Notebook

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/github/aws-samples/sample-mlops-bestpractices/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/2_deployment.ipynb)** — GitHub's renderer strips JavaScript that powers interactive output cells (Evidently reports, plotly charts, ipywidgets). nbviewer (run by Project Jupyter) renders them in full.


Deploy the trained model to a SageMaker serverless endpoint.

This notebook handles:
1. **Model Selection** — Choose from trained models in the Model Registry
2. **Endpoint Creation** — Deploy to a serverless endpoint with custom inference handler
3. **Endpoint Testing** — Verify the endpoint works correctly
4. **Endpoint Management** — Update or delete endpoints

**Prerequisites:**
- Run `1_training_pipeline_bank_marketing.ipynb` first to train a model
- Model must be registered in SageMaker Model Registry

**Custom Inference Handler:**
The endpoint uses a custom handler that automatically logs all predictions to Athena for drift monitoring.

**Environment:** SageMaker AI Notebook or local with AWS credentials.

## Setup

In [10]:
import os
import sys
import json
import boto3
from pathlib import Path
from dotenv import load_dotenv

# Find project root and add to path
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.config.config import (
    AWS_DEFAULT_REGION,
    SAGEMAKER_EXEC_ROLE,
    DATA_S3_BUCKET,
    ATHENA_DATABASE,
    ATHENA_OUTPUT_S3,
    MLFLOW_MODEL_NAME,
)
from src.utils.aws_utils import get_execution_role

# Deployment logic itself lives in src/train_pipeline/deploy_endpoint.py
# (also used by `main.py deploy`) — this notebook just calls into it so
# there's a single source of truth for the deployment steps.
from src.train_pipeline.deploy_endpoint import (
    resolve_inference_sqs_queue_url,
    build_inference_env,
    select_latest_approved_model,
    create_model_from_package,
    create_endpoint_config,
    deploy_endpoint,
    get_endpoint_status,
    delete_endpoint,
)

# Initialize clients
region = AWS_DEFAULT_REGION
sm_client = boto3.client('sagemaker', region_name=region)
runtime_client = boto3.client('sagemaker-runtime', region_name=region)
s3_client = boto3.client('s3', region_name=region)
sqs_client = boto3.client('sqs', region_name=region)

role = get_execution_role()

print(f'Region: {region}')
print(f'Role: {role}')
print(f'S3 Bucket: {DATA_S3_BUCKET}')
print(f'Athena DB: {ATHENA_DATABASE}')
print(f'MLflow Model: {MLFLOW_MODEL_NAME}')

Using SageMaker execution role from environment: arn:aws:iam::329430715989:role/fraud-detection-monitoring-SageMakerExecutionRole
Region: us-east-1
Role: arn:aws:iam::329430715989:role/fraud-detection-monitoring-SageMakerExecutionRole
S3 Bucket: fraud-detection-monitoring-data-329430715989
Athena DB: fraud_detection
MLflow Model: ml-model


## Deployment Configuration

In [11]:
# Endpoint configuration (overrides passed to the deploy_endpoint.py functions below).
# NOTE: ENDPOINT_NAME is 'fraud-detector-endpoint' for historical reasons (the CFN
# stack was deployed with ProjectName=fraud-detection-monitoring). The model behind
# it is trained on Bank Marketing data — the name is cosmetic.
ENDPOINT_NAME = 'fraud-detector-endpoint'
# MODEL_PACKAGE_GROUP matches the MLflow registered model name (from config.yaml).
# Pipeline.py registers each trained model into a SageMaker Model Package Group
# of the same name (pipeline.py:321), so the two registries stay in lockstep.
MODEL_PACKAGE_GROUP = MLFLOW_MODEL_NAME

# Serverless configuration
MEMORY_SIZE_MB = 2048  # 1024, 2048, 3072, 4096, 5120, 6144
MAX_CONCURRENCY = 10   # Max concurrent invocations

# Resolve the inference-logging SQS queue URL and build the custom inference
# handler's environment. CFN creates the queue (resource `InferenceLoggerQueue`,
# name `${ProjectName}-inference-logging`); the handler reads SQS_QUEUE_URL
# from its env and sends a message per prediction, and a downstream Lambda
# drains the queue into Athena `inference_responses`. If the queue can't be
# resolved, the endpoint still deploys but Athena logging is disabled.
INFERENCE_SQS_QUEUE_URL = resolve_inference_sqs_queue_url(sqs_client=sqs_client)
INFERENCE_ENV = build_inference_env(ENDPOINT_NAME, INFERENCE_SQS_QUEUE_URL, region)

print('Endpoint Configuration:')
print(f'  Name: {ENDPOINT_NAME}')
print(f'  Memory: {MEMORY_SIZE_MB} MB')
print(f'  Max Concurrency: {MAX_CONCURRENCY}')
print(f'\nAthena Logging: {INFERENCE_ENV["ENABLE_ATHENA_LOGGING"]}')
print(f'SQS Queue: {INFERENCE_SQS_QUEUE_URL or "(unresolved — logging disabled)"}')

[07/14/26 22:03:07] INFO     ✓ Resolved SQS queue:                                            ]8;id=5304005;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304006;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#87\87]8;;\
                             https://sqs.us-east-1.amazonaws.com/329430715989/fraud-detection                      
                             -monitoring-inference-logging                                                         

Endpoint Configuration:
  Name: fraud-detector-endpoint
  Memory: 2048 MB
  Max Concurrency: 10

Athena Logging: true
SQS Queue: https://sqs.us-east-1.amazonaws.com/329430715989/fraud-detection-monitoring-inference-logging


## 1. Select Model from Registry

Choose which model version to deploy.

In [12]:
# Select the latest approved model from the registry (set model_version=N to pin a specific one).
print(f'Available models in package group: {MODEL_PACKAGE_GROUP}\n')

model_info = select_latest_approved_model(MODEL_PACKAGE_GROUP, sm_client=sm_client)

LATEST_MODEL_ARN = model_info['arn']
LATEST_MODEL_VERSION = model_info['version']

print(f'✓ Will deploy model version {LATEST_MODEL_VERSION}')
print(f'  ARN: {LATEST_MODEL_ARN}')
print(f'  Created: {model_info["created_time"]}')

Available models in package group: ml-model



                    INFO     Listing approved models in package group: ml-model              ]8;id=5304011;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304012;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#168\168]8;;\

                    INFO     ✓ Selected model version 1:                                     ]8;id=5304017;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304018;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#212\212]8;;\
                             arn:aws:sagemaker:us-east-1:329430715989:model-package/ml-model                       
                             /1                                                                                    

✓ Will deploy model version 1
  ARN: arn:aws:sagemaker:us-east-1:329430715989:model-package/ml-model/1
  Created: 2026-07-14T21:31:43.910000+00:00


## 2. Create SageMaker Model with Inference Code

Package the model artifact with the custom inference handler.

In [13]:
# Creates the SageMaker Model FROM THE MODEL PACKAGE (not from raw Image+ModelDataUrl),
# so it carries ModelPackageName — needed for the drift Lambda's
# `describe_model → ModelPackageName` lookup to succeed (no fallback to
# `list_model_packages(SortOrder=Descending)`, and no risk of comparing current
# traffic to a baseline.json belonging to a different version). Also validates
# the artifact tarball contains code/inference.py and finalizes INFERENCE_ENV
# with the model version and script-mode vars (mutated in place).
model_name = create_model_from_package(
    LATEST_MODEL_ARN,
    LATEST_MODEL_VERSION,
    ENDPOINT_NAME,
    role,
    region,
    INFERENCE_ENV,
    sm_client=sm_client,
    s3_client=s3_client,
)

print(f'✓ Model created: {model_name}')
print(f'  ModelPackage: {LATEST_MODEL_ARN}')
print(f'  Version: {INFERENCE_ENV["MODEL_VERSION"]}')

                    INFO     Model package ARN  :                                            ]8;id=5304023;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304024;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#262\262]8;;\
                             arn:aws:sagemaker:us-east-1:329430715989:model-package/ml-model                       
                             /1                                                                                    

                    INFO     Model artifact     :                                            ]8;id=5304029;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304030;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#263\263]8;;\
                             s3://sagemaker-us-east-1-329430715989/fraud-detection/training/                       
                             output/pipelines-nsqw29wt5lkr-TrainModel-oRuO2ib4ix/output/mode                       
                             l.tar.gz                                                                              

                    INFO     Container image    :                                            ]8;id=5304035;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304036;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#264\264]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:                       
                             3.0-5                                                                                 

                    INFO     ✓ code/inference.py present in model artifact                   ]8;id=5304041;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304042;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#281\281]8;;\

                    INFO     Creating SageMaker model:                                       ]8;id=5304047;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304048;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#296\296]8;;\
                             fraud-detector-endpoint-model-1784066587...                                           

[07/14/26 22:03:08] INFO     ✓ Model created: fraud-detector-endpoint-model-1784066587       ]8;id=5304053;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304054;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#305\305]8;;\

                    INFO       ARN:                                                          ]8;id=5304059;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304060;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#306\306]8;;\
                             arn:aws:sagemaker:us-east-1:329430715989:model/fraud-detector-e                       
                             ndpoint-model-1784066587                                                              

✓ Model created: fraud-detector-endpoint-model-1784066587
  ModelPackage: arn:aws:sagemaker:us-east-1:329430715989:model-package/ml-model/1
  Version: v1


## 3. Create Endpoint Configuration

In [14]:
ENDPOINT_CONFIG_NAME = create_endpoint_config(
    ENDPOINT_NAME,
    model_name,
    memory_size_mb=MEMORY_SIZE_MB,
    max_concurrency=MAX_CONCURRENCY,
    sm_client=sm_client,
)

print(f'✓ Endpoint configuration created: {ENDPOINT_CONFIG_NAME}')

                    INFO     Creating endpoint configuration:                                ]8;id=5304065;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304066;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#336\336]8;;\
                             fraud-detector-endpoint-config-1784066588...                                          

                    INFO     ✓ Endpoint configuration created:                               ]8;id=5304071;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304072;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#350\350]8;;\
                             fraud-detector-endpoint-config-1784066588                                             

                    INFO       ARN:                                                          ]8;id=5304077;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304078;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#351\351]8;;\
                             arn:aws:sagemaker:us-east-1:329430715989:endpoint-config/fraud-                       
                             detector-endpoint-config-1784066588                                                   

✓ Endpoint configuration created: fraud-detector-endpoint-config-1784066588


## 4. Deploy Endpoint

Create or update the endpoint. This takes 5-10 minutes for serverless endpoints.

In [15]:
# Deploy the endpoint (idempotent). REDEPLOY_CLEAN=True deletes any existing
# endpoint first for a clean slate; set False to update the existing
# endpoint's config in place instead. This polls until the endpoint reaches
# InService/Failed, which takes 5-10 minutes for serverless endpoints —
# progress is logged as it goes.
REDEPLOY_CLEAN = True

result = deploy_endpoint(
    ENDPOINT_NAME,
    ENDPOINT_CONFIG_NAME,
    redeploy_clean=REDEPLOY_CLEAN,
    sm_client=sm_client,
)

print(f'\n✓ Endpoint status: {result["status"]}')
print(f'  Name: {result["endpoint_name"]}')
print(f'  ARN: {result["endpoint_arn"]}')

                    INFO     Endpoint fraud-detector-endpoint already exists (status:        ]8;id=5304084;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304085;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#405\405]8;;\
                             InService)                                                                            

                    INFO     Deleting existing endpoint fraud-detector-endpoint...           ]8;id=5304091;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304092;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#414\414]8;;\

[07/14/26 22:03:14] INFO     ✓ Endpoint fraud-detector-endpoint deleted                      ]8;id=5304098;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304099;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#422\422]8;;\

                    INFO     Creating endpoint: fraud-detector-endpoint...                   ]8;id=5304104;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304105;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#442\442]8;;\

                    INFO     ✓ Endpoint creation initiated                                   ]8;id=5304110;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304111;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#447\447]8;;\

...

[07/14/26 22:04:45] INFO     [90s] Endpoint status: Creating                                 ]8;id=5304146;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304147;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#466\466]8;;\

[07/14/26 22:05:15] INFO     [120s] Endpoint status: Creating                                ]8;id=5304152;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304153;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#466\466]8;;\

[07/14/26 22:05:45] INFO     [150s] Endpoint status: InService                               ]8;id=5304158;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304159;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#466\466]8;;\

                    INFO     ✓ Endpoint is live and ready for inference:                     ]8;id=5304164;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py\deploy_endpoint.py]8;;\:]8;id=5304165;file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/src/train_pipeline/deploy_endpoint.py#469\469]8;;\
                             fraud-detector-endpoint                                                               


✓ Endpoint status: InService
  Name: fraud-detector-endpoint
  ARN: arn:aws:sagemaker:us-east-1:329430715989:endpoint/fraud-detector-endpoint


## 5. Test Endpoint

Send a test prediction to verify the endpoint works.

In [16]:
# Test payload — Bank Marketing dataset (20 features matching dataset_schema.yaml)
# Categorical columns are label-encoded to integers during data download.
test_features = {
    # --- numeric features (as-is) ---
    'age': 41,
    'duration': 180,           # last-contact duration, seconds
    'campaign': 2,             # contacts this campaign
    'pdays': 999,              # days since last contact (999 = never)
    'previous': 0,             # contacts before this campaign
    'emp_var_rate': 1.1,       # employment variation rate
    'cons_price_idx': 93.994,  # consumer price index
    'cons_conf_idx': -36.4,    # consumer confidence index
    'euribor3m': 4.857,        # euribor 3-month rate
    'nr_employed': 5191.0,     # number of employees

    # --- label-encoded categoricals (integers) ---
    'job': 0,
    'marital': 1,
    'education': 6,
    'credit_default': 0,
    'housing': 2,
    'loan': 0,
    'contact': 1,
    'month': 6,
    'day_of_week': 1,
    'poutcome': 1,
}

print('Sending test prediction request (Bank Marketing payload)...')
print(f'Features: {json.dumps(test_features, indent=2)}\n')

try:
    response = runtime_client.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(test_features)
    )

    result = json.loads(response['Body'].read().decode())

    print('✓ Prediction successful!')
    print(f'\nRaw response: {json.dumps(result, indent=2)}')

    # The inference handler returns:
    #   {"predictions": [0/1], "probabilities": {"negative": [...], "positive": [...]}}
    # Older deployed versions may use "non_fraud"/"fraud" keys instead.
    pred = result['predictions'][0]
    probs = result.get('probabilities', {})
    prob_positive = (probs.get('positive') or probs.get('fraud', [None]))[0]

    label = "SUBSCRIBED" if pred == 1 else "NOT SUBSCRIBED"
    print(f'\nBank Marketing Result:')
    print(f'  Prediction: {label}')
    print(f'  P(subscribe): {prob_positive:.4f}' if prob_positive else '  P(subscribe): N/A')

except Exception as e:
    print(f'❌ Prediction failed: {e}')

Sending test prediction request (Bank Marketing payload)...
Features: {
  "age": 41,
  "duration": 180,
  "campaign": 2,
  "pdays": 999,
  "previous": 0,
  "emp_var_rate": 1.1,
  "cons_price_idx": 93.994,
  "cons_conf_idx": -36.4,
  "euribor3m": 4.857,
  "nr_employed": 5191.0,
  "job": 0,
  "marital": 1,
  "education": 6,
  "credit_default": 0,
  "housing": 2,
  "loan": 0,
  "contact": 1,
  "month": 6,
  "day_of_week": 1,
  "poutcome": 1
}



✓ Prediction successful!

Raw response: {
  "predictions": [
    0
  ],
  "probabilities": {
    "non_fraud": [
      0.9406411647796631
    ],
    "fraud": [
      0.05935882404446602
    ]
  }
}

Bank Marketing Result:
  Prediction: NOT SUBSCRIBED
  P(subscribe): 0.0594


In [17]:
# # ---------------------------------------------------------------------------
# # Legacy test payload — Kaggle Fraud Detection dataset (30 features)
# # ---------------------------------------------------------------------------
# # Uncomment this cell if you switch back to the fraud dataset. The model
# # must be retrained on fraud data first — sending these features to a Bank
# # Marketing model produces meaningless predictions.
#
# test_features_fraud = {
#     'transaction_hour': 14,
#     'transaction_day_of_week': 3,
#     'transaction_amount': 125.50,
#     'transaction_type_code': 1,
#     'customer_age': 35,
#     'customer_gender': 1,
#     'customer_tenure_months': 24,
#     'account_age_days': 730,
#     'distance_from_home_km': 5.2,
#     'distance_from_last_transaction_km': 2.1,
#     'time_since_last_transaction_min': 120,
#     'online_transaction': 1,
#     'international_transaction': 0,
#     'high_risk_country': 0,
#     'merchant_category_code': 5411,
#     'merchant_reputation_score': 0.85,
#     'chip_transaction': 1,
#     'pin_used': 1,
#     'card_present': 1,
#     'cvv_match': 1,
#     'address_verification_match': 1,
#     'num_transactions_24h': 2,
#     'num_transactions_7days': 8,
#     'avg_transaction_amount_30days': 95.30,
#     'max_transaction_amount_30days': 250.00,
#     'velocity_score': 0.3,
#     'recurring_transaction': 0,
#     'previous_fraud_incidents': 0,
#     'credit_limit': 5000.0,
#     'available_credit_ratio': 0.75,
# }
#
# print('Sending test prediction request (Kaggle fraud payload)...')
# response = runtime_client.invoke_endpoint(
#     EndpointName=ENDPOINT_NAME,
#     ContentType='application/json',
#     Body=json.dumps(test_features_fraud),
# )
# result = json.loads(response['Body'].read().decode())
# print(f'\nRaw response: {json.dumps(result, indent=2)}')
#
# pred = result['predictions'][0]
# probs = result.get('probabilities', {})
# prob_fraud = (probs.get('positive') or probs.get('fraud', [None]))[0]
# print(f'\nFraud Detection Result:')
# print(f'  Prediction: {"FRAUD" if pred == 1 else "NOT FRAUD"}')
# print(f'  P(fraud): {prob_fraud:.4f}' if prob_fraud else '  P(fraud): N/A')

## 6. Endpoint Management

Utility functions for managing the endpoint.

In [18]:
# get_endpoint_status() and delete_endpoint() are imported from
# src/train_pipeline/deploy_endpoint.py (see Setup cell above) — no local
# wrapper definitions needed.

print('Management functions available:')
print('  - get_endpoint_status(endpoint_name, sm_client=sm_client)')
print('  - delete_endpoint(endpoint_name, sm_client=sm_client)')
print(f'\nExample: get_endpoint_status("{ENDPOINT_NAME}", sm_client=sm_client)')

Management functions available:
  - get_endpoint_status(endpoint_name, sm_client=sm_client)
  - delete_endpoint(endpoint_name, sm_client=sm_client)

Example: get_endpoint_status("fraud-detector-endpoint", sm_client=sm_client)


## Next Steps

Now that your endpoint is deployed:

1. **Run Inference** → Go to `3_inference_monitoring.ipynb` to:
   - Send predictions to the endpoint
   - Monitor inference performance
   - Detect data and model drift

2. **View Metrics** → Go to `4_governance_dashboard.ipynb` to:
   - View model performance over time
   - Analyze drift patterns
   - Generate governance reports

3. **Monitor Endpoint** → AWS Console:
   - CloudWatch metrics for invocations and latency
   - CloudWatch logs for detailed request/response logging
   - Athena for querying logged predictions